In [30]:
from pathlib import Path
import os
import re
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from esda.moran import Moran
from IPython.display import display
from libpysal.weights import KNN
from scipy.stats import chi2
from spreg import GM_Error, ML_Lag

warnings.filterwarnings("ignore")
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mpl-cache"))
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

df      = gpd.read_parquet("cleaned_datasets/resale_with_all_features_dummies.parquet")
df_raw  = gpd.read_parquet("cleaned_datasets/resale_with_all_features.parquet")

# Set MAX_ROWS = None to run the full dataset
MAX_ROWS = 3000 # only for sample test 
RANDOM_STATE = 42
K_NEIGHBORS = 8
SPATIAL_METHOD = "ord"


### 1. Data prep for OLS and Spatial

In [31]:
# 1. Parse remaining_lease to numeric 
def parse_remaining_lease(val):
    val = str(val).strip()
    if "year" in val:
        parts  = val.split()
        years  = int(parts[0])
        months = int(parts[2]) if len(parts) >= 4 else 0
        return years + months / 12
    return float(val)

df["remaining_lease_numeric"] = df["remaining_lease"].apply(parse_remaining_lease)

# 2. Log-transform target 
assert (df["resale_price"] > 0).all(), "Non-positive prices found"
df["log_resale_price"] = np.log(df["resale_price"])

# 3. Cap walking_dist_mall_m outlier 
# max of 20,885m is a geocoding error
cap_mall = df["walking_dist_mall_m"].quantile(0.99)
n_capped = (df["walking_dist_mall_m"] > cap_mall).sum()
df["walking_dist_mall_m"] = df["walking_dist_mall_m"].clip(upper=cap_mall)

# 4. Treatment variable
# binary: 1 if within 1km of Tier-1 school (MOE Phase 2B cutoff)
df["near_tier1_1km"] = (df["num_tier1_schools_1km"] >= 1).astype(int)

# 5. Additional OLS features
# remaining_lease has nonlinear effect (convex below ~60 years), squared term captures this for linear models
df["remaining_lease_sq"] = df["remaining_lease_numeric"] ** 2

# 6. Drop non-features
non_features_drop = [
    "month",            
    "year",                
    "remaining_lease",  
    "resale_price",     
]

# 7. df_ols (OLS / Ridge / LASSO / Elastic Net)
df = df.drop(
    columns=non_features_drop,
    errors="ignore"
).copy()

df_ols = df.copy()
df_spatial = df.copy()

### 2. OLS Regression

- This section is the same as 'regression.ipynb' since residuals are required for Moran I' test

In [32]:
target       = "log_resale_price"
mean_price   = np.exp(df_ols[target].mean())
y            = df_ols[target]

# 1. BINARISE SCHOOL VARIABLES
# For generic schools: if count >= 1, then near = 1
df_ols["near_schools_1km"] = (df_ols["num_schools_1km"] >= 1).astype(int)
df_ols["near_schools_2km"] = (df_ols["num_schools_2km"] >= 1).astype(int)
df_ols["near_tier1_2km"]   = (df_ols["nearest_tier1_primary_school_dist_m"] <= 2000).astype(int)
df_ols["near_normalschools_1km"] = (df_ols["nearest_normal_primary_school_dist_m"] <= 1000).astype(int)
df_ols["near_normalschools_2km"] = (df_ols["nearest_normal_primary_school_dist_m"] <= 2000).astype(int)

# 2. DEFINE FEATURES
flat_controls = [
    "floor_area_sqm", "remaining_lease_numeric",
    # Flat Types
    "flat_type_2 ROOM", "flat_type_3 ROOM", "flat_type_4 ROOM",
    "flat_type_5 ROOM", "flat_type_EXECUTIVE",
    # Storey Ranges
    "storey_range_04 TO 06", "storey_range_07 TO 09", "storey_range_10 TO 12",
    "storey_range_13 TO 15", "storey_range_16 TO 18", "storey_range_19 TO 21",
    "storey_range_22 TO 24", "storey_range_25 TO 27", "storey_range_28 TO 30",
    "storey_range_31 TO 33", "storey_range_34 TO 36", "storey_range_37 TO 39",
    "storey_range_40 TO 42", "storey_range_43 TO 45", "storey_range_46 TO 48",
    "storey_range_49 TO 51",
    # Flat Models
    "flat_model_3Gen", "flat_model_Adjoined flat", "flat_model_Apartment",
    "flat_model_DBSS", "flat_model_Improved", "flat_model_Improved-Maisonette",
    "flat_model_Maisonette", "flat_model_Model A", "flat_model_Model A-Maisonette",
    "flat_model_Model A2", "flat_model_New Generation", "flat_model_Premium Apartment",
    "flat_model_Premium Apartment Loft", "flat_model_Premium Maisonette",
    "flat_model_Simplified", "flat_model_Standard", "flat_model_Terrace",
    "flat_model_Type S1", "flat_model_Type S2"
]

loc_controls = ["walking_dist_mrt_m",
    "walking_dist_busstop_m", "walking_dist_hawker_m", "walking_dist_mall_m",
    "dist_cbd_m"]

town_controls = [
# Towns
    "town_BEDOK", "town_BISHAN", "town_BUKIT BATOK", "town_BUKIT MERAH",
    "town_BUKIT PANJANG", "town_BUKIT TIMAH", "town_CENTRAL AREA",
    "town_CHOA CHU KANG", "town_CLEMENTI", "town_GEYLANG", "town_HOUGANG",
    "town_JURONG EAST", "town_JURONG WEST", "town_KALLANG/WHAMPOA",
    "town_MARINE PARADE", "town_PASIR RIS", "town_PUNGGOL", "town_QUEENSTOWN",
    "town_SEMBAWANG", "town_SENGKANG", "town_SERANGOON", "town_TAMPINES",
    "town_TOA PAYOH", "town_WOODLANDS", "town_YISHUN",
]


In [33]:
#COMPARING SCHOOLS VS GOOD
# MODEL SPECS
MAIN_MODEL_VAR = {
    "Model 1 (1km) baseline": ["near_tier1_1km"],
    "Model 2 (1km) add flat controls": ["near_tier1_1km"],
    "Model 3 (1km) add location controls": ["near_tier1_1km"],
    "Model 4 (1km) add town controls": ["near_tier1_1km"],
    "Model 5 (1km) add school controls": ["near_schools_1km", "near_tier1_1km"],

    "Model 1 (2km) baseline": ["near_tier1_2km"],
    "Model 2 (2km) add flat controls": ["near_tier1_2km"],
    "Model 3 (2km) add location controls": ["near_tier1_2km"],
    "Model 4 (2km) add town controls": ["near_tier1_2km"],
    "Model 5 (2km) add school controls": ["near_schools_2km", "near_tier1_2km"],

    "Model 5 (only near school 1km)": ["near_schools_1km"],
    "Model 6 (only near school 2km)": ["near_schools_2km"]
}

# OLS
all_model_objects = {} # needed for diagnostics of spatial models later
results = {}

for model_label, school_vars in MAIN_MODEL_VAR.items():
    if "baseline" in model_label:
        feature_cols = school_vars
    elif "flat controls" in model_label:
        feature_cols = flat_controls + school_vars
    elif "location controls" in model_label:
        feature_cols = flat_controls + loc_controls + school_vars
    else:
        feature_cols = flat_controls + loc_controls + town_controls + school_vars
    X = sm.add_constant(df_ols[feature_cols])
    
    mod = sm.OLS(y, X).fit(cov_type="HC3")

    # store model for spatial diagnostics later
    all_model_objects[model_label] = {
        "ols_model": mod,
        "X_with_constant_model": X,
        "y_series_model": y_series,
    }
    
    for var in school_vars:
        beta = mod.params[var]
        pval = mod.pvalues[var]
        results[(model_label, var)] = {
            "beta": beta,
            "se": mod.bse[var],
            "p": pval,
            "sig": ("***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else "n.s."),
            "pct": (np.exp(beta) - 1) * 100,
            "adj_r2": mod.rsquared_adj,
            "n": int(mod.nobs)
        }

# SUMMARY
print(f"{'='*100}")
print(f"{'OLS RESULTS: HEDONIC PRICE PREMIUMS':^100}")
print(f"{'='*100}")
header = f"{'Model Spec':<20} {'Variable':<20} {'Beta':>10} {'p-val':>8} {'Sig':>5} {'% Effect':>12} {'Adj. R2':>10}"
print(header)
print("-" * 100)

for (model_label, var), r in results.items():
    print(f"{model_label:<20} {var:<20} {r['beta']:>10.4f} {r['p']:>8.4f} {r['sig']:>5} {r['pct']:>+11.2f}% {r['adj_r2']:>10.4f}")

print("-" * 100)

                                OLS RESULTS: HEDONIC PRICE PREMIUMS                                 
Model Spec           Variable                   Beta    p-val   Sig     % Effect    Adj. R2
----------------------------------------------------------------------------------------------------
Model 1 (1km) baseline near_tier1_1km           0.0601   0.0000   ***       +6.19%     0.0056
Model 2 (1km) add flat controls near_tier1_1km           0.1223   0.0000   ***      +13.01%     0.7702
Model 3 (1km) add location controls near_tier1_1km           0.0319   0.0000   ***       +3.24%     0.8891
Model 4 (1km) add town controls near_tier1_1km           0.0003   0.6173  n.s.       +0.03%     0.9254
Model 5 (1km) add school controls near_schools_1km        -0.0546   0.0000   ***       -5.32%     0.9262
Model 5 (1km) add school controls near_tier1_1km           0.0045   0.0000   ***       +0.46%     0.9262
Model 1 (2km) baseline near_tier1_2km           0.0210   0.0000   ***       +2.13%     0.

### 3. Spatial Regression preparation
- This section prepares data for spatial regression modeling of housing prices, where location matters. 
- The LM tests will help determine whether a spatial lag model (price influenced by neighboring prices) or spatial error model (spatial structure in the errors) is more appropriate for the data.

### 3.1 Helper functions to compute LM tests and extract spatial regression results

In [34]:
def pad_array(values, target_length):
    """
    Pad a list of values with NaN to reach a target length.
    """
    arr = np.asarray(values, dtype=float).reshape(-1)
    if len(arr) >= target_length:
        return arr[:target_length]
    return np.pad(arr, (0, target_length - len(arr)), constant_values=np.nan)


def extract_spreg_results(model, model_label, focus_vars):
    """
    Extract only selected variables from a spreg spatial regression model.
    """
    betas = np.asarray(model.betas).reshape(-1)
    names = list(getattr(model, "name_x", [])) + ["lambda"]

    # stats
    stat_entries = getattr(model, "z_stat", None)
    if stat_entries is not None:
        p_values = np.array([entry[1] for entry in stat_entries])
    else:
        p_values = np.repeat(np.nan, len(betas))

    df = pd.DataFrame({
        "variable": names,
        "beta": betas,
        "p": p_values
    })

    # keep only variables of interest
    df = df[df["variable"].isin(focus_vars)].copy()

    # formatting
    df["sig"] = df["p"].apply(
        lambda p: "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else "n.s."
    )
    df["pct"] = (np.exp(df["beta"]) - 1) * 100

    df["model"] = model_name
    df["adj_r2"] = getattr(model, "pr2", np.nan)

    return df[["model", "variable", "beta", "p", "sig", "pct", "adj_r2"]]


def compute_lm_tests(ols_model, X_with_constant, y_series, w):
    """ 
    Compute Robust LM-lag, and Robust LM-error tests based on the OLS model residuals and spatial weights.
    """
    residuals = ols_model.resid.to_numpy().reshape(-1, 1)
    fitted = ols_model.fittedvalues.to_numpy().reshape(-1, 1)
    y_array = y_series.to_numpy().reshape(-1, 1)
    x_array = X_with_constant.to_numpy()
    n = len(y_array)
    xtxi = np.asarray(ols_model.normalized_cov_params)
    sigma2_n = ((residuals.T @ residuals) / n).item()

    w_sparse = w.sparse
    wu = w_sparse @ residuals
    utwu_ds = ((residuals.T @ wu) / sigma2_n).item()
    utwy_ds = ((residuals.T @ (w_sparse @ y_array)) / sigma2_n).item()
    t_term = float(np.sum((((w_sparse.T + w_sparse) @ w_sparse).diagonal())))

    wxb = w_sparse @ fitted
    xwxb = x_array.T @ wxb
    j_num = (wxb.T @ wxb) - (xwxb.T @ xtxi @ xwxb) + (t_term * sigma2_n)
    j_term = (j_num / (n * sigma2_n)).item()

    robust_lm_lag = float(((utwy_ds - utwu_ds) ** 2) / ((n * j_term) - t_term))
    robust_lm_error = float(((utwu_ds - ((t_term * utwy_ds) / (n * j_term))) ** 2) / (
        t_term * (1.0 - (t_term / (n * j_term))))
    )

    return pd.DataFrame(
        [
            {"test": "Robust LM-lag", "statistic": robust_lm_lag, "p_value": chi2.sf(robust_lm_lag, 1)},
            {"test": "Robust LM-error", "statistic": robust_lm_error, "p_value": chi2.sf(robust_lm_error, 1)},
        ]
    )

In [35]:
target       = "log_resale_price"
mean_price   = np.exp(df_spatial[target].mean())
y            = df_spatial[target]

# 1. BINARISE SCHOOL VARIABLES
# For generic schools: if count >= 1, then near = 1
df_spatial["near_schools_1km"] = (df_spatial["num_schools_1km"] >= 1).astype(int)
df_spatial["near_schools_2km"] = (df_spatial["num_schools_2km"] >= 1).astype(int)
df_spatial["near_tier1_2km"]   = (df_spatial["nearest_tier1_primary_school_dist_m"] <= 2000).astype(int)
df_spatial["near_normalschools_1km"] = (df_spatial["nearest_normal_primary_school_dist_m"] <= 1000).astype(int)
df_spatial["near_normalschools_2km"] = (df_spatial["nearest_normal_primary_school_dist_m"] <= 2000).astype(int)

# 2. DEFINE FEATURES
flat_controls = [
    "floor_area_sqm", "remaining_lease_numeric",
    # Flat Types
    "flat_type_2 ROOM", "flat_type_3 ROOM", "flat_type_4 ROOM",
    "flat_type_5 ROOM", "flat_type_EXECUTIVE",
    # Storey Ranges
    "storey_range_04 TO 06", "storey_range_07 TO 09", "storey_range_10 TO 12",
    "storey_range_13 TO 15", "storey_range_16 TO 18", "storey_range_19 TO 21",
    "storey_range_22 TO 24", "storey_range_25 TO 27", "storey_range_28 TO 30",
    "storey_range_31 TO 33", "storey_range_34 TO 36", "storey_range_37 TO 39",
    "storey_range_40 TO 42", "storey_range_43 TO 45", "storey_range_46 TO 48",
    "storey_range_49 TO 51",
    # Flat Models
    "flat_model_3Gen", "flat_model_Adjoined flat", "flat_model_Apartment",
    "flat_model_DBSS", "flat_model_Improved", "flat_model_Improved-Maisonette",
    "flat_model_Maisonette", "flat_model_Model A", "flat_model_Model A-Maisonette",
    "flat_model_Model A2", "flat_model_New Generation", "flat_model_Premium Apartment",
    "flat_model_Premium Apartment Loft", "flat_model_Premium Maisonette",
    "flat_model_Simplified", "flat_model_Standard", "flat_model_Terrace",
    "flat_model_Type S1", "flat_model_Type S2"
]

loc_controls = ["walking_dist_mrt_m",
    "walking_dist_busstop_m", "walking_dist_hawker_m", "walking_dist_mall_m",
    "dist_cbd_m"]

town_controls = [
# Towns
    "town_BEDOK", "town_BISHAN", "town_BUKIT BATOK", "town_BUKIT MERAH",
    "town_BUKIT PANJANG", "town_BUKIT TIMAH", "town_CENTRAL AREA",
    "town_CHOA CHU KANG", "town_CLEMENTI", "town_GEYLANG", "town_HOUGANG",
    "town_JURONG EAST", "town_JURONG WEST", "town_KALLANG/WHAMPOA",
    "town_MARINE PARADE", "town_PASIR RIS", "town_PUNGGOL", "town_QUEENSTOWN",
    "town_SEMBAWANG", "town_SENGKANG", "town_SERANGOON", "town_TAMPINES",
    "town_TOA PAYOH", "town_WOODLANDS", "town_YISHUN",
]


In [36]:
# X and y
X = sm.add_constant(df_spatial[feature_cols].astype(float), has_constant="add")
y = df_spatial["log_resale_price"].astype(float).to_numpy().reshape(-1, 1)

# spatial weights
coords = np.column_stack([
    df_spatial.geometry.x.values,
    df_spatial.geometry.y.values
])

w = KNN.from_array(coords, k=K_NEIGHBORS)
w.transform = "r"

In [37]:
all_model_diagnostics = {}

for model_name, model_parts in all_model_objects.items():
    print(f"\n===== Diagnostics: {model_name} =====")

    ols_model = model_parts["ols_model"]
    X_with_constant_model = model_parts["X_with_constant_model"]
    y_series_model = model_parts["y_series_model"]

    # Moran's I on OLS residuals
    # Flattening ensures the input is a 1D array for Moran
    resids = ols_model.resid.to_numpy().flatten()
    residual_moran = Moran(resids, w, permutations=0)

    moran_df = pd.DataFrame(
        [
            {
                "model": model_name,
                "test": "Moran's I on OLS residuals",
                "statistic": residual_moran.I,
                "p_value": residual_moran.p_norm,
            }
        ]
    )

    # LM diagnostics
    lm_results_df = compute_lm_tests(
        ols_model,
        X_with_constant_model,
        y_series_model,
        w
    )
    
    lm_results_df["model"] = model_name
    diagnostics_df = pd.concat([moran_df, lm_results_df], ignore_index=True)
    all_model_diagnostics[model_name] = diagnostics_df

    display(diagnostics_df)


===== Diagnostics: Model 1 (1km) baseline =====


,model,test,statistic,p_value
0,Model 1 (1km) baseline,Moran's I on OLS residuals,0.825154,0.000000
1,Model 1 (1km) baseline,Robust LM-lag,0.094684,0.758305
2,Model 1 (1km) baseline,Robust LM-error,6.033988,0.014033



===== Diagnostics: Model 2 (1km) add flat controls =====


,model,test,statistic,p_value
0,Model 2 (1km) add flat controls,Moran's I on OLS residuals,0.792694,0.000000
1,Model 2 (1km) add flat controls,Robust LM-lag,"2,117.646577",0.000000
2,Model 2 (1km) add flat controls,Robust LM-error,"418,894.545781",0.000000



===== Diagnostics: Model 3 (1km) add location controls =====


...

,model,test,statistic,p_value
0,Model 5 (2km) add school controls,Moran's I on OLS residuals,0.522014,0.000000
1,Model 5 (2km) add school controls,Robust LM-lag,936.849651,0.000000
2,Model 5 (2km) add school controls,Robust LM-error,"233,485.963038",0.000000



===== Diagnostics: Model 5 (only near school 1km) =====


,model,test,statistic,p_value
0,Model 5 (only near school 1km),Moran's I on OLS residuals,0.523508,0.000000
1,Model 5 (only near school 1km),Robust LM-lag,949.173833,0.000000
2,Model 5 (only near school 1km),Robust LM-error,"234,669.398806",0.000000



===== Diagnostics: Model 6 (only near school 2km) =====


,model,test,statistic,p_value
0,Model 6 (only near school 2km),Moran's I on OLS residuals,0.523181,0.000000
1,Model 6 (only near school 2km),Robust LM-lag,918.923953,0.000000
2,Model 6 (only near school 2km),Robust LM-error,"234,566.102404",0.000000


- Since Robust LM-error > Robust LM-lag, use spatial error model (SEM)

### 3. Spaital Error Model

### ---- Original 2 models ----

In [38]:
SEM_MODEL_VAR = {
    "Model 1 (1km)": ["near_schools_1km", "near_tier1_1km"],
    "Model 2 (2km)": ["near_schools_2km", "near_tier1_2km"],
}

school_model_vars = [
    "near_schools_1km", "near_tier1_1km",
    "near_schools_2km", "near_tier1_2km"
]

base_feature_cols = [
    col for col in feature_cols
    if col not in school_model_vars
]

sem_results = {}

for model_label, school_vars in SEM_MODEL_VAR.items():

    # Model-specific features
    this_feature_cols = base_feature_cols + school_vars

    X = df_ols[this_feature_cols].astype(float)
    y_np = np.asarray(y).reshape(-1, 1)

    sem_model = GM_Error(
        y_np,
        X.to_numpy(),
        w=w,
        name_x=this_feature_cols
    )

    # extract coefficients
    betas = np.asarray(sem_model.betas).reshape(-1)
    names = this_feature_cols + ["lambda"]
    pvals = np.array([stat[1] for stat in sem_model.z_stat])

    lambda_val = betas[-1]

    for var in school_vars:
        idx = names.index(var)

        beta = betas[idx]
        pval = pvals[idx]

        sem_results[(model_label, var)] = {
            "beta": beta,
            "p": pval,
            "sig": ("***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else "n.s."),
            "pct": (np.exp(beta) - 1) * 100,
            "pseudo_r2": getattr(sem_model, "pr2", np.nan),
            "lambda": lambda_val,
            "n": sem_model.n
        }
        
print(f"{'='*100}")
print(f"{'SPATIAL ERROR MODEL (SEM): HEDONIC PRICE PREMIUMS':^100}")
print(f"{'='*100}")

header = f"{'Model Spec':<20} {'Variable':<20} {'Beta':>10} {'p-val':>8} {'Sig':>5} {'% Effect':>12} {'Pseudo R2':>10} {'Lambda':>10}"
print(header)
print("-" * 100)

for (model_label, var), r in sem_results.items():
    print(
        f"{model_label:<20} {var:<20} "
        f"{r['beta']:>10.4f} {r['p']:>8.4f} {r['sig']:>5} "
        f"{r['pct']:>+11.2f}% {r['pseudo_r2']:>10.4f} {r['lambda']:>10.4f}"
    )

print("-" * 100)

                         SPATIAL ERROR MODEL (SEM): HEDONIC PRICE PREMIUMS                          
Model Spec           Variable                   Beta    p-val   Sig     % Effect  Pseudo R2     Lambda
----------------------------------------------------------------------------------------------------
Model 1 (1km)        near_schools_1km        -0.0293   0.0000   ***       -2.89%     0.9223     0.7080
Model 1 (1km)        near_tier1_1km          -0.0506   0.0000   ***       -4.93%     0.9223     0.7080
Model 2 (2km)        near_schools_2km        -0.0104   0.0535     *       -1.04%     0.9225     0.7071
Model 2 (2km)        near_tier1_2km          -0.6850   0.0000   ***      -49.59%     0.9225     0.7071
----------------------------------------------------------------------------------------------------


### ---- Consec models ----

In [39]:
sem_results = {}

for model_label, school_vars in MAIN_MODEL_VAR.items():

    if "baseline" in model_label:
        feature_cols = school_vars
    elif "flat controls" in model_label:
        feature_cols = flat_controls + school_vars
    elif "location controls" in model_label:
        feature_cols = flat_controls + loc_controls + school_vars
    else:
        feature_cols = flat_controls + loc_controls + town_controls + school_vars

    X = df_ols[feature_cols].astype(float)
    y_np = y.reshape(-1, 1)

    sem_model = GM_Error(
        y_np,
        X.to_numpy(),
        w=w,
        name_x=feature_cols
    )

    # extract coefficients
    betas = np.asarray(sem_model.betas).reshape(-1)
    names = feature_cols + ["lambda"]

    pvals = np.array([stat[1] for stat in sem_model.z_stat])

    # lambda is last element
    lambda_val = betas[-1]

    for var in school_vars:
        idx = names.index(var)

        beta = betas[idx]
        pval = pvals[idx]

        sem_results[(model_label, var)] = {
            "beta": beta,
            "p": pval,
            "sig": ("***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else "n.s."),
            "pct": (np.exp(beta) - 1) * 100,
            "pseudo_r2": getattr(sem_model, "pr2", np.nan),
            "lambda": lambda_val,
            "n": sem_model.n
        }

print(f"{'='*100}")
print(f"{'SPATIAL ERROR MODEL (SEM): HEDONIC PRICE PREMIUMS':^100}")
print(f"{'='*100}")

header = f"{'Model Spec':<20} {'Variable':<20} {'Beta':>10} {'p-val':>8} {'Sig':>5} {'% Effect':>12} {'Pseudo R2':>10} {'Lambda':>10}"
print(header)
print("-" * 100)

for (model_label, var), r in sem_results.items():
    print(
        f"{model_label:<20} {var:<20} "
        f"{r['beta']:>10.4f} {r['p']:>8.4f} {r['sig']:>5} "
        f"{r['pct']:>+11.2f}% {r['pseudo_r2']:>10.4f} {r['lambda']:>10.4f}"
    )

print("-" * 100)

                         SPATIAL ERROR MODEL (SEM): HEDONIC PRICE PREMIUMS                          
Model Spec           Variable                   Beta    p-val   Sig     % Effect  Pseudo R2     Lambda
----------------------------------------------------------------------------------------------------
Model 1 (1km) baseline near_tier1_1km          13.3489   0.0000   *** +62715329.16%     0.0056     0.8624
Model 2 (1km) add flat controls near_tier1_1km           0.5331   0.0000   ***      +70.42%     0.7447     0.8408
Model 3 (1km) add location controls near_tier1_1km          -0.0000   0.0000   ***       -0.00%     0.8810     0.7820
Model 4 (1km) add town controls near_tier1_1km          -0.0242   0.0000   ***       -2.39%     0.9215     0.7106
Model 5 (1km) add school controls near_schools_1km        -0.0293   0.0000   ***       -2.89%     0.9223     0.7080
Model 5 (1km) add school controls near_tier1_1km          -0.0506   0.0000   ***       -4.93%     0.9223     0.7080
Model 1 (2k